In [700]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torch.utils.data import TensorDataset
import os
from collections import defaultdict
import torch.nn.functional as F
import numpy as np
import math

In [701]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

data_folder = "./data/shakespeare-dataset-main/text"
text = ""
for root, _, files in os.walk(data_folder):
    for file in files:
        if file.endswith(".txt"):
            path = os.path.join(root, file)
            with open(path, 'r', encoding='utf-8') as f:
                text += f.read()

In [702]:
vocab = {}
vo = 0
for let in text:
    if let not in vocab.keys():
        vocab[let] = vo
        vo += 1 

flocab = {}
for i,let in enumerate(vocab.keys()):
    flocab[let] = (i/len(vocab.keys()))

special_token_num = 1
vocab_size = len(vocab.keys()) + 1
rev_vocab = {idx: token for token, idx in vocab.items()}

In [703]:
def deflocab(seq):
    building = ""
    for x in seq:
        for j, (let,fl) in enumerate(flocab.items()):
                if fl >= x:
                    building += let
                    break
    return building

def devocab(seq):
    building = ""
    for x in seq:
        if(x != 84):
            building += rev_vocab[int(x)]
    return building
     

In [704]:
floaty_text = []
tokenized_text = []
building_sample = []
building_sample2 = []

for let in text:
    building_sample.append(flocab[let])
    building_sample2.append(vocab[let])
    if(let == '\n'):
        floaty_text.append(torch.tensor(building_sample))
        tokenized_text.append(building_sample2)
        building_sample = []
        building_sample2 = []

In [705]:
def corrupt_tokens(x: torch.Tensor, t: int, num_steps: int, vocab_size: int) -> torch.Tensor:
    corruption_prob = t / num_steps
    corruption_prob_tm1 = (t-1) / num_steps

    rx = torch.rand(x.shape).to(device)
    mask = (rx < corruption_prob).to(device)
    mask2 = (rx < corruption_prob_tm1).to(device)
    random_tokens = torch.randint_like(x, 0, vocab_size)
    random_tokens2 = torch.randint_like(x, 0, vocab_size)

    corrupted = torch.where(mask, random_tokens, x)
    corrupted_tm1 = torch.where(mask2, random_tokens2, x)

    return corrupted,corrupted_tm1

In [706]:
class BucketedBatchDataset(Dataset):
    def __init__(self, token_lists, batch_size=64, bucket_size=5, pad_token=vocab_size-1):
        """
        Args:
            token_lists (List[List[int]]): list of tokenized sequences as lists of ints
            batch_size (int): number of samples per batch
            bucket_size (int): bucket width for sequence lengths
            pad_token (int): token ID used for padding
        """
        self.batches = []
        self.pad_token = pad_token

        # Step 1: Bucket sequences by length range
        buckets = defaultdict(list)
        for seq in token_lists:
            length = len(seq)
            bucket_min = (length // bucket_size) * bucket_size
            bucket_max = bucket_min + bucket_size

            buckets[(bucket_min, bucket_max)].append(seq)

        # Step 2: For each bucket, pad and batch sequences
        for (bucket_min, bucket_max), sequences in buckets.items():
            max_len = bucket_max
            # Pad sequences to max_len and convert to tensors
            padded_seqs = []
            for seq in sequences:
                padding_len = max_len - len(seq)
                padded_seq = seq + [pad_token] * padding_len
                padded_seqs.append(torch.tensor(padded_seq, dtype=torch.long))

            # Create batches
            for i in range(0, len(padded_seqs), batch_size):
                batch = torch.stack(padded_seqs[i:i+batch_size])
                self.batches.append(batch)

    def __len__(self):
        return len(self.batches)

    def __getitem__(self, idx):
        return self.batches[idx]
    
shakespeare = BucketedBatchDataset(tokenized_text)

In [707]:
def relative_position_bucket(rel_pos, num_buckets=32, max_distance=128, bidirectional=True):
    n = rel_pos
    if bidirectional:
        half = num_buckets // 2
        sign = (n < 0).to(torch.long)
        n = n.abs()
    else:
        half = 0
        sign = 0
        n = n.clamp(min=0)

    max_exact = num_buckets // 2
    is_small = n < max_exact
    val = torch.where(
        is_small,
        n,
        max_exact + (
            (torch.log(n.float() / max_exact + 1e-6) /
             math.log(max_distance / max_exact + 1e-6)) *
            (num_buckets - max_exact)
        ).to(torch.long).clamp(max=num_buckets - max_exact - 1)
    )
    return val + sign * half

In [708]:
class MultiheadAttentionWithRelBias(nn.Module):
    def __init__(self, embed_dim, num_heads, use_out_proj=True):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.use_out_proj = use_out_proj

        assert self.head_dim * num_heads == embed_dim, "embed_dim must be divisible by num_heads"

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)

        if use_out_proj:
            self.out_proj = nn.Linear(embed_dim, embed_dim)

        # T5-style: Learnable relative bias weights for each head.
        # This layer will project the 'embed_dim' of your rel_bias_emb
        # to 'num_heads' scalar biases, effectively learning how each head
        # should interpret the rich relative positional information.
        self.relative_attention_bias = nn.Linear(embed_dim, num_heads)

    def forward(self, x, rel_bias_emb):
        # x: (batch_size, seq_len, embed_dim)
        # rel_bias_emb: (seq_len, seq_len, embed_dim)
        # This rel_bias_emb should ideally contain features for each (query_pos, key_pos) relative distance.
        batch_size, seq_len, _ = x.size()

        Q = self.q_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim)
        K = self.k_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim)
        V = self.v_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim)

        Q = Q.transpose(1, 2)  # (B, H, L, D_h)
        K = K.transpose(1, 2)  # (B, H, L, D_h)
        V = V.transpose(1, 2)  # (B, H, L, D_h)

        # Compute attention scores
        # (B, H, L, D_h) @ (B, H, D_h, L) -> (B, H, L, L)
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)

        # Apply T5-style relative attention bias
        # rel_bias_emb: (L, L, E) -> project to (L, L, H) -> permute to (H, L, L) -> unsqueeze to (1, H, L, L)
        # This linear layer effectively learns the scalar bias for each head from the rich rel_bias_emb features.
        relative_bias = self.relative_attention_bias(rel_bias_emb) # (L, L, H)
        relative_bias = relative_bias.permute(2, 0, 1).unsqueeze(0) # (1, H, L, L)

        # Add the relative bias to the attention scores (logits)
        attn_scores = attn_scores + relative_bias # Broadcasted (B, H, L, L)

        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_output = torch.matmul(attn_weights, V)  # (B, H, L, D_h)

        # Concatenate heads and apply final linear layer
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.embed_dim)

        if self.use_out_proj:
            attn_output = self.out_proj(attn_output)  # (B, L, E)

        return attn_output

In [ ]:
num_timesteps = 50
class ConvDiffAtt(nn.Module):
    def __init__(self,embedding_dim=512,max_spatial=7,vocab_size=85,num_timesteps=num_timesteps,pos_bucket_num=64,max_bucket_dist=128):
        super().__init__()

        self.embedding_dim = embedding_dim
        self.max_spatial = max_spatial
        self.pos_bucket_num = pos_bucket_num
        self.max_bucket_dist=max_bucket_dist

        self.embedding_table_vocab = nn.Embedding(vocab_size,embedding_dim=self.embedding_dim)

        self.embedding_table_timesteps = nn.Embedding(num_timesteps,embedding_dim=self.embedding_dim)

        self.embedding_rel_buckets = nn.Embedding(pos_bucket_num,embedding_dim=self.embedding_dim)

        #self.pos_emb = nn.Embedding(max_seq_len, embedding_dim)

        self.embedding_convs = nn.ModuleList()

        local_out_channels = self.embedding_dim // ((self.max_spatial+1)//2)

        for i in range(1, self.max_spatial+1,2):
            conv = nn.Conv1d(
                in_channels=self.embedding_dim,
                out_channels=local_out_channels,
                kernel_size=i,
                stride=1,
                padding=(i-1)//2  # no internal padding, we do manual padding in forward
            )
            self.embedding_convs.append(conv)

        self.embed_conv_ln = nn.LayerNorm((self.embedding_dim))

        self.attend = MultiheadAttentionWithRelBias(embed_dim=self.embedding_dim,num_heads=8)
        
        self.att_layernm = nn.LayerNorm((self.embedding_dim))

        self.gloop = nn.Sequential(
            nn.Linear(self.embedding_dim,1024),
            nn.GELU(),
            nn.Linear(1024,1024),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.LayerNorm((1024)),
            nn.Linear(1024,vocab_size),
        )

    def conv_embed(self,x):
        out = []
        for conv in self.embedding_convs:
            out.append(conv(x))

            #print(conv(a).shape)
        return self.embed_conv_ln((torch.cat(out,dim=1)).transpose(2,1)).transpose(1,2)
    
    def rel_buckets(self,rel_pos_mat):
        a = relative_position_bucket(rel_pos_mat,self.pos_bucket_num,self.max_bucket_dist)
        return a


    def forward(self, x, t):
        rel_pos_mat = torch.abs(torch.arange(0,x.shape[-1]).unsqueeze(0) - torch.arange(0,x.shape[-1]).unsqueeze(0).T).to(device)
        rel_pos_mat_embedded = self.embedding_rel_buckets(rel_pos_mat)

        embd = self.embedding_table_vocab(x)

        p = embd.transpose(1,2)
        pc = self.conv_embed(p).transpose(2,1)
        
        attended = self.attend(pc,rel_pos_mat_embedded)

        a = self.att_layernm(attended + self.embedding_table_timesteps(t) + embd)
        
        return self.gloop(a)

model = ConvDiffAtt(vocab_size=vocab_size).to(device)

print(f"Num-Params : {sum([x.numel() for x in model.parameters()])}")

#print(shakespeare[0].unsqueeze(-1).shape)
ax = torch.tensor(0).to(device)
model(shakespeare[3].to(device),ax).shape

Num-Params : 3871837


torch.Size([64, 25, 85])

In [735]:
learning_rate = 1e-4
epochs = 10
loss = nn.CrossEntropyLoss(ignore_index=84,label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

def train_loop(dataset, model, loss_fn, optimizer):
    size = len(tokenized_text)
    curr = 0
    model.train()

    for batch, X in enumerate(dataset):
        rand_t = torch.randint(0,num_timesteps+1,(1,))[0].to(device)
        dv = X.to(device)
        
        r,t = corrupt_tokens(dv,rand_t,num_timesteps,vocab_size-special_token_num)
        
        pred = model(r,rand_t)

        logits_flat = pred.view(-1, pred.size(-1))  # [1600, 84]
        targets_flat = t.view(-1) 
        loss = loss_fn(logits_flat,targets_flat)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        curr += X.shape[0]
        if batch % 512 == 0:
            loss = loss.item()
            print(f"loss: {loss:>7f}  [{curr:>5d}/{size:>5d}]")

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(shakespeare, model, loss, optimizer)

Epoch 1
-------------------------------
loss: 1.805290  [   64/182729]
loss: 2.016294  [32774/182729]
loss: 0.804184  [65489/182729]
loss: 4.197224  [98129/182729]
loss: 4.022951  [130810/182729]
loss: 4.295222  [163498/182729]
Epoch 2
-------------------------------
loss: 0.821914  [   64/182729]
loss: 2.914331  [32774/182729]
loss: 4.407964  [65489/182729]
loss: 3.521791  [98129/182729]
loss: 3.103938  [130810/182729]
loss: 3.726271  [163498/182729]
Epoch 3
-------------------------------
loss: 1.902876  [   64/182729]
loss: 3.937968  [32774/182729]
loss: 1.364330  [65489/182729]
loss: 4.249057  [98129/182729]
loss: 3.418918  [130810/182729]
loss: 3.694733  [163498/182729]
Epoch 4
-------------------------------
loss: 3.491940  [   64/182729]
loss: 3.264546  [32774/182729]
loss: 1.848718  [65489/182729]
loss: 4.036417  [98129/182729]
loss: 1.435524  [130810/182729]
loss: 4.021979  [163498/182729]
Epoch 5
-------------------------------
loss: 2.880485  [   64/182729]
loss: 4.019114  [

In [737]:
actu = shakespeare[0][0].unsqueeze(1).to(device)

rnd = torch.randint_like(actu,0,vocab_size).to(device)
#rnd = torch.full_like(actu,84)
for t in reversed(range(1, num_timesteps+1)):
    print(repr(f"{t} {devocab(rnd)}"))
    logits = model(rnd, torch.tensor(t).to(device))
    
    probs = torch.softmax(logits, dim=-1)
    rnd = torch.multinomial(probs.squeeze(-2),1)



print(rnd.squeeze(-1))
devocab(rnd)

'50 ?mvZaN2E6n]Gd Zgz!0b")P0’'
'49 ,46FFP;xy0..5C=5N3/\nhzx2D'
'48 "2:-hlEgf]JVKo=]3o9U7Wc3>'
"47 f!N]tW/s0’3U]Z2Rk:4Z6I1a'"
'46 YHC2Q[bX:<1:E\t\'U.lJ)jB,H"'
'45 lfI1E7d<TqsKbDIu.)Z::Cx1H'
'44 FUC’36"Vk4Qarl\ngHqE9V6pQ.'
'43  ";[o5mHH((yqFQN3kp7\t/tk0'
"42 Kdn&Ixko\nN&D8ti'QV)pUCea/"
'41 vsA?!kn,HrtIP?m,&8]b)12cq'
"40 nZ8xX='RNQ83'-QF-C’v2BA[8"
'39 :8qMg=sdXf\nM3XIGDAxHYb!0m'
'38 hVY ==.!qch=dQe=HA&!FWu-u'
"37 -oHa=/]rB0&=2'e=ZIYX,sa;m"
'36 Z8):Vnd6\t,o=Gv8=Vd6X’:Fj;'
'35 stRm Fut’>"=S -=.MMf:9\'tD'
'34 t48wefml! KVpH8=u8PXI.eON'
'33 vE1\tPQiv’IcG3oI=oqM.i;2V='
'32 3p6:?Or4eG?MF,?=wR1wR:>e='
'31 Ui-I71A9L"Emu=enf[?N)1rm='
'30  iJqdl4Oa0w\nd=X=FkYKCPYX='
'29 nD]"P"XvQHIkO=e=GGC9fF/O='
'28 sZtVCY"TWM1wW=e=g!C?3TQ[='
'27 ElYI[6M/\nU1sP=l=-k>UrfB]='
'26 ]D]zIgOPdvWdc=Q=:hgrsShs='
'25 St!pjz6FkTU8(uT=ehm-"x.W='
'24 7,!xRVF.up4UT!q=!hw-ag5S='
'23 \nq;yB=B38xf4f4f=?htn=OQ-='
'22 UFj->=/EJM.z"A;=QIML=gt-='
'21 pouS)<q;gmXb.Abn;mt’=uo-='
'20 A05Ss8(lHmq8lEAdb6]y=5>9='
"19 ME3S!?e6YjIoMSUj!OkL='

'OhH!\nJD!(!03\nC="->\n N=gl='